In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder
from sklearn.utils.class_weight import compute_class_weight
from tqdm import tqdm

# ══════════════════════════════════════════════════════════════════════════════
# ADVERSARIAL TRAINING (QSLP approach) for HybridResNet v3 / ClassicalResNet v3
# ══════════════════════════════════════════════════════════════════════════════
#
# WHAT THIS FILE DOES:
#   Implements the full QSLP training pipeline from the paper:
#   "Adversarially Robust QNN for Malware Detection"
#   adapted for your Virus-MNIST / ResNet-based architecture.
#
# THREE DEFENCES COMBINED (from Algorithm 1 in the paper):
#   1. Clean loss          — normal cross-entropy on real images
#   2. FGSM adversarial    — pixel-space attack: single-step gradient sign
#   3. PGD adversarial     — pixel-space attack: iterative projected gradient
#   (QNI-CCP is the 4th component; it requires forward access to the
#    feature extractor, which is handled via a hook below)
#
# LOSS FORMULA (from paper, Algorithm 1 lines 19-21):
#   loss = 0.50 * loss_clean
#        + 0.15 * loss_qni     ← QNI-CCP (latent-space perturbation)
#        + 0.10 * loss_fgsm    ← FGSM pixel-space adversarial
#        + 0.15 * loss_pgd     ← PGD pixel-space adversarial
#        + 0.10 * centroid_reg ← centroid regularisation (pulls features to class mean)
#
# ATTACK PARAMETERS (from paper Table 3):
#   FGSM ε = 0.03
#   PGD  ε = 0.10, step size = 0.01, iterations = 7
#
# HOW TO USE:
#   1. Train your HybridResNet v3 or ClassicalResNet v3 normally first.
#   2. Load the saved checkpoint.
#   3. Run this file to fine-tune with adversarial training.
#   → Alternatively, train from scratch by skipping the checkpoint load.
# ══════════════════════════════════════════════════════════════════════════════


# ─────────────────────────────────────────────
# PASTE YOUR FULL MODEL DEFINITION HERE
# ─────────────────────────────────────────────
# Copy-paste your HybridResNet (with PennyLane) OR ClassicalResNet here.
# This file uses ClassicalResNet as an example since it runs without PennyLane.
# The adversarial training logic is IDENTICAL for both — just swap the class.
# ─────────────────────────────────────────────

def seed_all(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_all(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────
n_qubits     = 8
q_out_dim    = 3 * n_qubits     # 24
batch_size   = 32
num_classes  = 10
num_epochs   = 40               # adversarial fine-tuning; fewer epochs than full training
lr           = 0.0002           # lower LR for fine-tuning (was 0.0005)
weight_decay = 3e-4

# ADVERSARIAL ATTACK HYPERPARAMETERS (from paper Table 3)
FGSM_EPS    = 0.03              # single-step perturbation budget
PGD_EPS     = 0.10              # maximum total perturbation (ℓ∞)
PGD_ALPHA   = 0.01              # step size per PGD iteration
PGD_ITERS   = 7                 # number of PGD iterations
EPSILON_Q   = 1.0               # QNI-CCP scaling factor (α=1.0, β=1.0 from paper)

# LOSS WEIGHTS (from paper Algorithm 1)
W_CLEAN     = 0.50              # dominant: keep clean accuracy
W_QNI       = 0.15              # QNI-CCP: latent-space robustness
W_FGSM      = 0.10              # FGSM: lightweight pixel-space adversarial
W_PGD       = 0.15              # PGD: stronger pixel-space adversarial
W_CENTROID  = 0.10              # centroid reg: pull features to class mean

# ─────────────────────────────────────────────
# TRANSFORMS & DATASETS (unchanged from v3)
# ─────────────────────────────────────────────
train_transform = transforms.Compose([
    transforms.Grayscale(1),
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])
eval_transform = transforms.Compose([
    transforms.Grayscale(1),
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

TRAIN_PATH = 'virus_MNIST dataset/tain'
TEST_PATH  = 'virus_MNIST dataset/test'
VAL_PATH   = 'virus_MNIST dataset/val'

try:
    train_dataset = ImageFolder(TRAIN_PATH, transform=train_transform)
    test_dataset  = ImageFolder(TEST_PATH,  transform=eval_transform)
    val_dataset   = ImageFolder(VAL_PATH,   transform=eval_transform)
    print("Datasets loaded successfully")
except Exception as e:
    print(f"Error loading datasets: {e}")

try:
    labels = [label for _, label in train_dataset.samples]
    class_weights = compute_class_weight(
        class_weight='balanced', classes=np.unique(labels), y=labels
    )
    class_weight_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)
    print("Class weights computed:", class_weights)
except Exception as e:
    print(f"Could not compute class weights: {e}. Using uniform weights.")
    class_weight_tensor = torch.ones(num_classes).to(device)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                          num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False,
                          num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False,
                          num_workers=4, pin_memory=True)


# ══════════════════════════════════════════════════════════════════════════════
# SECTION 1: FEATURE EXTRACTOR HOOK
# ══════════════════════════════════════════════════════════════════════════════
#
# WHY WE NEED THIS:
#   QNI-CCP operates in the LATENT SPACE — the 64-dim feature vector AFTER
#   the backbone (stem+stage1+stage2+stage3+GAP) but BEFORE the bridge.
#   We need to extract this intermediate representation without rebuilding
#   the model. PyTorch hooks let us intercept activations mid-forward-pass.
#
# HOW IT WORKS:
#   A "forward hook" is a callback that PyTorch calls automatically whenever
#   a specific layer finishes its forward pass. We attach one to model.gap,
#   which is the last backbone layer (Global Average Pool).
#   The hook stores the output (B, 64) into a shared container.
# ══════════════════════════════════════════════════════════════════════════════

class FeatureHook:
    """
    Registers a forward hook on a target layer to capture its output.
    Usage:
        hook = FeatureHook(model.gap)
        _ = model(x)            # run forward pass
        features = hook.output  # (B, 64) after GAP
        hook.remove()           # clean up when done
    """
    def __init__(self, layer):
        self.output = None
        # register_forward_hook: called after layer.forward() with (module, input, output)
        self._handle = layer.register_forward_hook(self._capture)

    def _capture(self, module, input, output):
        # output shape after GAP: (B, 64, 1, 1) → we flatten later
        self.output = output

    def remove(self):
        self._handle.remove()   # deregister hook to avoid memory leaks


def get_backbone_features(model, x):
    """
    Runs a forward pass and returns the flattened GAP output (B, 64).
    This is the feature space where QNI-CCP and centroid reg operate.

    Data flow explained:
      x (B,1,32,32) → stem → stage1 → stage2 → stage3 → GAP → (B,64,1,1)
      → view → (B, 64)   ← this is what we return
    """
    hook = FeatureHook(model.gap)   # attach hook to GAP layer
    with torch.no_grad():
        _ = model(x)                # trigger forward pass (hook fires during this)
    features = hook.output.view(hook.output.size(0), -1)  # (B,64,1,1) → (B,64)
    hook.remove()
    return features


# ══════════════════════════════════════════════════════════════════════════════
# SECTION 2: CLASS CENTROID COMPUTATION
# ══════════════════════════════════════════════════════════════════════════════
#
# WHAT IS A CENTROID:
#   For each class c, the centroid μ_c is the MEAN of all 64-dim feature
#   vectors that belong to class c in the training set.
#   μ_c = (1/N_c) * Σ z_i  for all i where y_i == c
#
# WHY CENTROIDS:
#   They represent the "ideal centre" of each class in feature space.
#   QNI-CCP uses them to push features TOWARD the wrong class centroid
#   during training, forcing the model to build better decision margins.
#
# WHEN TO RECOMPUTE:
#   Every 5 epochs (from paper Algorithm 1 line 3-5).
#   Recomputing every epoch is wasteful; centroids shift slowly.
# ══════════════════════════════════════════════════════════════════════════════

def compute_class_centroids(model, dataloader, device, num_classes):
    """
    Computes the mean feature vector (centroid) for each class.

    Returns:
        centroids: tensor of shape (num_classes, 64)
                   centroids[c] = mean latent vector for class c

    Data flow:
      For each batch → get_backbone_features → (B, 64)
      → accumulate per class → divide by count → centroid per class
    """
    model.eval()
    # accumulators: sum of feature vectors and count per class
    sum_features = torch.zeros(num_classes, 64, device=device)   # (C, 64)
    count        = torch.zeros(num_classes, device=device)        # (C,)

    with torch.no_grad():
        for x, y in tqdm(dataloader, desc="Computing centroids", leave=False):
            x, y = x.to(device), y.to(device)
            feats = get_backbone_features(model, x)   # (B, 64)
            # scatter-add: for each sample i, add feats[i] to sum_features[y[i]]
            for c in range(num_classes):
                mask = (y == c)                       # boolean mask for class c
                if mask.sum() > 0:
                    sum_features[c] += feats[mask].sum(dim=0)  # accumulate
                    count[c]        += mask.sum()

    # avoid division by zero for classes with no samples in this batch
    count = count.clamp(min=1.0)
    centroids = sum_features / count.unsqueeze(1)     # (C, 64)
    return centroids


# ══════════════════════════════════════════════════════════════════════════════
# SECTION 3: QNI-CCP — Quantum Noise Injection Class Center Perturbation
# ══════════════════════════════════════════════════════════════════════════════
#
# CONCEPT (from paper Section 3.3):
#   Standard adversarial training perturbs PIXELS. But after downsampling
#   through the CNN backbone, pixel perturbations barely affect the latent
#   representation. QNI-CCP directly perturbs the LATENT FEATURES instead.
#
# THREE STEPS:
#   Step 1 — Feature Gradient Sensitivity (S):
#     S = ∇_z L(f(z), y)  → which features most influence the loss?
#     High |S_i| = feature i is critical; perturbing it has maximum impact.
#
#   Step 2 — Target Class Selection (c'):
#     Pick a random WRONG class c' ≠ y for each sample.
#     Get the centroid μ_{c'} of that wrong class.
#
#   Step 3 — Sensitivity-Weighted Perturbation:
#     Δ_base = μ_{c'} - z           (direction from true features to wrong centroid)
#     δ      = S ⊙ Δ_base           (Hadamard: weight by sensitivity)
#     z'     = z + ε_q * δ          (perturbed features)
#
#   EFFECT: Features are pushed toward the wrong class centroid,
#   with more push on the sensitive features. Model must learn to resist this.
# ══════════════════════════════════════════════════════════════════════════════

def qni_ccp_perturb(model, x, y, centroids, epsilon_q=1.0):
    """
    Computes QNI-CCP perturbed features in latent space.

    Args:
        model      : trained model with backbone + bridge + q_layer + classifier
        x          : input batch (B, 1, 32, 32)
        y          : true labels (B,)
        centroids  : (num_classes, 64) class centroids from compute_class_centroids
        epsilon_q  : QNI-CCP scaling factor (paper uses 1.0)

    Returns:
        perturbed_features : (B, 64) — features shifted toward wrong class centroid

    Data flow:
        x → backbone → z (B,64)
        z → loss wrt z → sensitivity S (B,64)
        y → pick wrong class c' → get centroid μ_{c'} (B,64)
        perturbed = z + ε_q * (S ⊙ (μ_{c'} - z))
    """
    model.eval()

    # ── Step 1: Get features with gradient tracking ──────────────────
    hook = FeatureHook(model.gap)
    output = model(x)                               # full forward pass
    z = hook.output.view(hook.output.size(0), -1)   # (B, 64) — latent features
    hook.remove()

    # Enable grad on z so we can compute ∂L/∂z
    z = z.detach().requires_grad_(True)

    # Recompute classifier path from z (bypass backbone since z is already extracted)
    # This is equivalent to: bridge(z) → q_layer → classifier
    z_bridge = model.bridge(z)                      # (B, 64) → (B, 16)
    z_q      = model.q_layer(z_bridge)              # (B, 16) → (B, 24)
    logits   = model.classifier(z_q)               # (B, 24) → (B, 10)

    loss = F.cross_entropy(logits, y)               # scalar loss
    loss.backward()                                 # compute ∂L/∂z

    # S = gradient of loss w.r.t. features — shape (B, 64)
    S = z.grad.detach()
    # Normalise S to prevent scale explosion (ℓ2 norm per sample)
    S_norm = S / (S.norm(dim=1, keepdim=True) + 1e-8)  # (B, 64)

    # ── Step 2: Select random wrong target class per sample ──────────
    original_features = z.detach()                  # (B, 64) — detach from graph
    target_classes = []
    for i in range(y.size(0)):
        available = [c for c in range(centroids.size(0)) if c != y[i].item()]
        # randomly pick one wrong class
        chosen = available[torch.randint(0, len(available), (1,)).item()]
        target_classes.append(chosen)
    target_class = torch.tensor(target_classes, device=y.device)  # (B,)

    mu_c_prime = centroids[target_class]            # (B, 64) — wrong class centroids

    # ── Step 3: Sensitivity-weighted perturbation ────────────────────
    delta_base   = mu_c_prime - original_features   # direction toward wrong centroid (B,64)
    delta_weighted = S_norm * delta_base             # ⊙ Hadamard: focus on sensitive features
    perturbed    = original_features + epsilon_q * delta_weighted  # (B, 64)

    return perturbed.detach()


# ══════════════════════════════════════════════════════════════════════════════
# SECTION 4: PIXEL-SPACE ATTACKS — FGSM AND PGD
# ══════════════════════════════════════════════════════════════════════════════
#
# WHY PIXEL-SPACE ATTACKS:
#   QNI-CCP defends at the latent level. But real attackers modify PIXELS.
#   FGSM and PGD generate adversarial images that are perceptually similar
#   to originals but fool the classifier. Training on these images teaches
#   the model to correctly classify perturbed inputs.
#
# FGSM (Fast Gradient Sign Method) — Goodfellow et al. 2014:
#   x_adv = x + ε * sign(∇_x L(model(x), y))
#   Single step in the direction that maximises loss.
#   Fast but weak; serves as a lightweight regulariser.
#
# PGD (Projected Gradient Descent) — Madry et al. 2017:
#   Iterative FGSM with projection back to ε-ball around original image.
#   x_0 = x + uniform noise ∈ [-ε, ε]  (random start)
#   for k iterations:
#     x_{k+1} = Π_{x±ε}[ x_k + α * sign(∇_x L(model(x_k), y)) ]
#   Stronger than FGSM; considered the gold standard adversarial attack.
# ══════════════════════════════════════════════════════════════════════════════

def fgsm_attack(model, images, labels, eps_fgsm=0.03):
    """
    Generates FGSM adversarial examples.

    Args:
        model      : neural network (used in eval mode)
        images     : clean batch (B, 1, 32, 32), values in [-1, 1]
        labels     : true labels (B,)
        eps_fgsm   : perturbation budget ε (paper: 0.03)

    Returns:
        images_adv : adversarial images (B, 1, 32, 32), still in [-1, 1]

    Data flow:
        images → model → loss → ∂L/∂images → sign → ε * sign → add to images
        Result: image that maximises loss in one step
    """
    model.eval()
    images_adv = images.clone().detach().requires_grad_(True)  # need grad on input
    labels     = labels.to(images.device)

    logits = model(images_adv)                      # forward pass
    loss   = F.cross_entropy(logits, labels)        # loss at clean image
    model.zero_grad()
    loss.backward()                                 # compute ∂L/∂images_adv

    # move one step in the sign of the gradient → maximises loss
    perturbation = eps_fgsm * images_adv.grad.sign()   # (B,1,32,32), values ±eps_fgsm
    images_adv   = images_adv + perturbation
    images_adv   = torch.clamp(images_adv, min=-1.0, max=1.0)   # stay in valid range
    return images_adv.detach()


def pgd_attack(model, images, labels,
               pgd_eps=0.10, pgd_alpha=0.01, pgd_iters=7):
    """
    Generates PGD adversarial examples.

    Args:
        model      : neural network (used in eval mode)
        images     : clean batch (B, 1, 32, 32), values in [-1, 1]
        labels     : true labels (B,)
        pgd_eps    : maximum total perturbation ε (paper: 0.10)
        pgd_alpha  : step size per iteration (paper: 0.01)
        pgd_iters  : number of iterations (paper: 7)

    Returns:
        images_adv : adversarial images (B, 1, 32, 32), constrained to ε-ball

    Data flow:
        Start with images + random noise ∈ [-ε, ε]
        Repeat pgd_iters times:
            → model → loss → gradient sign → take step of size α
            → project back to ε-ball around original (clamp delta to ε)
            → clamp absolute values to [-1, 1]
        Stronger adversary: explores more of the ε-neighbourhood
    """
    model.eval()
    images_orig = images.clone().detach()

    # Random start: initialise perturbation uniformly in [-ε, ε]
    images_adv = images_orig + torch.empty_like(images_orig).uniform_(-pgd_eps, pgd_eps)
    images_adv = torch.clamp(images_adv, -1.0, 1.0).detach()   # valid image range

    for _ in range(pgd_iters):
        images_adv.requires_grad_(True)             # fresh grad each step
        logits = model(images_adv)
        loss   = F.cross_entropy(logits, labels)
        model.zero_grad()
        loss.backward()

        # gradient step toward maximum loss
        step       = pgd_alpha * images_adv.grad.sign()     # (B,1,32,32)
        images_adv = images_adv.detach() + step

        # project: total perturbation must stay within ε-ball
        delta      = torch.clamp(images_adv - images_orig, min=-pgd_eps, max=pgd_eps)
        images_adv = torch.clamp(images_orig + delta, -1.0, 1.0).detach()

    return images_adv


# ══════════════════════════════════════════════════════════════════════════════
# SECTION 5: ADVERSARIAL TRAINING LOOP
# ══════════════════════════════════════════════════════════════════════════════
#
# COMBINED LOSS (Algorithm 1 from paper):
#   loss = 0.50 * CE(model(x_clean), y)
#        + 0.15 * CE(classifier(q_layer(bridge(z_qni))), y)   ← QNI-CCP
#        + 0.10 * CE(model(x_fgsm), y)                        ← FGSM
#        + 0.15 * CE(model(x_pgd), y)                         ← PGD
#        + 0.10 * ‖z_clean - centroid[y]‖²                   ← centroid reg
#
# WHY EACH WEIGHT:
#   0.50 clean   → preserve base accuracy; this is still the main objective
#   0.15 QNI-CCP → meaningful latent-space defence, but not dominant
#   0.10 FGSM    → lightweight adversarial signal, lower weight (FGSM is weaker)
#   0.15 PGD     → stronger adversarial signal, same weight as QNI-CCP
#   0.10 centroid→ latent regularisation; prevents feature collapse
#
# CENTROID UPDATE SCHEDULE:
#   Recompute every 5 epochs. Centroids drift as the model learns;
#   stale centroids give increasingly inaccurate perturbation targets.
# ══════════════════════════════════════════════════════════════════════════════

def train_adversarial_epoch(model, train_loader, optimizer, centroids, device,
                             fgsm_eps=FGSM_EPS, pgd_eps=PGD_EPS,
                             pgd_alpha=PGD_ALPHA, pgd_iters=PGD_ITERS,
                             epsilon_q=EPSILON_Q):
    """
    Single epoch of QSLP adversarial training.

    Steps per batch:
      1. Compute 5 losses (clean, QNI-CCP, FGSM, PGD, centroid)
      2. Combine with fixed weights
      3. Backprop + gradient clip + optimizer step

    Returns:
        epoch_loss : average combined loss over all batches
        epoch_acc  : accuracy on CLEAN inputs (proxy for base performance)
    """
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    for x, y in tqdm(train_loader, desc="Adv Training", leave=False):
        x, y = x.to(device), y.to(device)

        # ── Loss 1: Clean loss ──────────────────────────────────────────
        # Standard CE on unperturbed inputs.
        # Weight 0.50: dominant term — clean accuracy must not collapse.
        logits_clean = model(x)                             # (B, 10) logits
        loss_clean   = F.cross_entropy(logits_clean, y)    # scalar

        # ── Loss 2: QNI-CCP (latent-space perturbation) ─────────────────
        # Perturb features in the 64-dim backbone output space.
        # Then pass perturbed features through bridge → q_layer → classifier.
        # Teaches the model to classify correctly even when features are pushed
        # toward wrong-class centroids.
        # Weight 0.15: important but secondary to clean loss.
        perturbed_z  = qni_ccp_perturb(model, x, y, centroids, epsilon_q)
        # perturbed_z: (B, 64) — already in backbone output space
        z_bridge     = model.bridge(perturbed_z)            # (B, 64) → (B, 16)
        z_q          = model.q_layer(z_bridge)              # (B, 16) → (B, 24)
        logits_qni   = model.classifier(z_q)               # (B, 24) → (B, 10)
        loss_qni     = F.cross_entropy(logits_qni, y)      # scalar

        # ── Loss 3: FGSM adversarial loss ──────────────────────────────
        # Generate adversarial images via single-step gradient sign attack.
        # Pass them through the model and compute CE loss.
        # Weight 0.10: lightweight adversarial signal.
        x_fgsm       = fgsm_attack(model, x, y, eps_fgsm=fgsm_eps)  # (B,1,32,32)
        model.train()                   # fgsm_attack sets model.eval(); restore
        logits_fgsm  = model(x_fgsm)                       # (B, 10)
        loss_fgsm    = F.cross_entropy(logits_fgsm, y)     # scalar

        # ── Loss 4: PGD adversarial loss ───────────────────────────────
        # Generate adversarial images via iterative PGD attack (stronger).
        # Weight 0.15: stronger attack gets higher weight than FGSM.
        x_pgd        = pgd_attack(model, x, y,
                                   pgd_eps=pgd_eps, pgd_alpha=pgd_alpha,
                                   pgd_iters=pgd_iters)    # (B,1,32,32)
        model.train()                   # pgd_attack sets model.eval(); restore
        logits_pgd   = model(x_pgd)                        # (B, 10)
        loss_pgd     = F.cross_entropy(logits_pgd, y)      # scalar

        # ── Loss 5: Centroid regularisation ────────────────────────────
        # Pulls the clean backbone features toward their class centroid.
        # Reduces feature variance → more compact, robust representations.
        # Weight 0.10: auxiliary regulariser.
        z_clean       = get_backbone_features(model, x)    # (B, 64) clean features
        centroid_reg  = ((z_clean - centroids[y]) ** 2).mean()  # mean squared distance

        # ── Combined loss (Algorithm 1, lines 19-21) ───────────────────
        loss = (W_CLEAN    * loss_clean   +   # 0.50
                W_QNI      * loss_qni     +   # 0.15
                W_FGSM     * loss_fgsm    +   # 0.10
                W_PGD      * loss_pgd     +   # 0.15
                W_CENTROID * centroid_reg)     # 0.10  → sums to 1.00

        # ── Backpropagation ────────────────────────────────────────────
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # prevent explosion
        optimizer.step()

        total_loss += loss.item()
        correct    += (logits_clean.argmax(1) == y).sum().item()  # clean acc tracking
        total      += y.size(0)

    return total_loss / len(train_loader), correct / total


def evaluate(model, dataloader, device):
    """
    Standard evaluation on clean data.
    Returns (loss, accuracy).

    Data flow: images → model → argmax → compare to labels
    """
    model.eval()
    total_loss, correct, total = 0.0, 0, 0

    with torch.no_grad():
        for x, y in dataloader:
            x, y       = x.to(device), y.to(device)
            logits     = model(x)
            loss       = F.cross_entropy(logits, y)
            total_loss += loss.item()
            correct    += (logits.argmax(1) == y).sum().item()
            total      += y.size(0)

    return total_loss / len(dataloader), correct / total


def evaluate_adversarial(model, dataloader, device,
                          fgsm_eps=FGSM_EPS,
                          pgd_eps=PGD_EPS, pgd_alpha=PGD_ALPHA, pgd_iters=PGD_ITERS):
    """
    Evaluates model accuracy under FGSM and PGD attacks.
    Used at end of training to report adversarial robustness.

    Returns:
        fgsm_acc : accuracy on FGSM-attacked images
        pgd_acc  : accuracy on PGD-attacked images

    Data flow:
        For each batch: generate x_fgsm and x_pgd → run model → compute accuracy
    """
    model.eval()
    fgsm_correct, pgd_correct, total = 0, 0, 0

    for x, y in tqdm(dataloader, desc="Adversarial Eval", leave=False):
        x, y = x.to(device), y.to(device)

        # FGSM evaluation
        x_fgsm       = fgsm_attack(model, x, y, eps_fgsm=fgsm_eps)
        model.eval()
        with torch.no_grad():
            logits_fgsm  = model(x_fgsm)
            fgsm_correct += (logits_fgsm.argmax(1) == y).sum().item()

        # PGD evaluation
        x_pgd        = pgd_attack(model, x, y,
                                   pgd_eps=pgd_eps, pgd_alpha=pgd_alpha,
                                   pgd_iters=pgd_iters)
        with torch.no_grad():
            logits_pgd   = model(x_pgd)
            pgd_correct  += (logits_pgd.argmax(1) == y).sum().item()

        total += y.size(0)

    return fgsm_correct / total, pgd_correct / total


# ══════════════════════════════════════════════════════════════════════════════
# SECTION 6: MAIN TRAINING SCRIPT
# ══════════════════════════════════════════════════════════════════════════════

# ── STEP 1: Build or load model ───────────────────────────────────────────────
# OPTION A: Load pretrained weights and fine-tune adversarially (RECOMMENDED)
#   Uncomment the relevant block below based on whether you're using
#   HybridResNet (with quantum layer) or ClassicalResNet.

# ── For HybridResNet v3 (requires PennyLane + model definition from hybrid file) ──
# from hybrid_resnet_v3 import HybridResNet  # or paste the class above
# model = HybridResNet(n_qubits=n_qubits, q_out_dim=q_out_dim,
#                      num_classes=num_classes, dropout=0.35).to(device)
# checkpoint = torch.load("hybrid_resnet_v3.pth", map_location=device)
# model.load_state_dict(checkpoint['model_state_dict'])
# print(f"Loaded HybridResNet v3. Starting val acc: {checkpoint['val_acc']:.4f}")

# ── For ClassicalResNet v3 (no quantum dependencies) ──────────────────────────
# from classical_resnet_v3 import ClassicalResNet
# model = ClassicalResNet(n_qubits=n_qubits, q_out_dim=q_out_dim,
#                         num_classes=num_classes, dropout=0.35).to(device)
# checkpoint = torch.load("classical_resnet_v3.pth", map_location=device)
# model.load_state_dict(checkpoint['model_state_dict'])
# print(f"Loaded ClassicalResNet v3. Starting val acc: {checkpoint['val_acc']:.4f}")

# OPTION B: Train from scratch (paste model class and instantiate directly)
# model = YourModel(...).to(device)

# ── STEP 2: Optimizer (AdamW with lower LR for fine-tuning) ──────────────────
# Lower LR than initial training (0.0002 vs 0.0005) because:
#   - Pretrained weights are already in a good region
#   - Aggressive LR during adversarial fine-tuning degrades clean accuracy
optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

# ReduceLROnPlateau: halve LR if train loss doesn't improve for 5 epochs
# More conservative than OneCycleLR for fine-tuning
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=5, factor=0.5, verbose=True
)

# ── STEP 3: Initial centroid computation ─────────────────────────────────────
print("Computing initial class centroids...")
centroids = compute_class_centroids(model, train_loader, device, num_classes)
# centroids: (10, 64) — mean backbone features per class

# ── STEP 4: Training loop ─────────────────────────────────────────────────────
best_val_acc               = 0.0
early_stopping_patience    = 12
epochs_without_improvement = 0
train_losses, val_losses   = [], []
train_accs,   val_accs     = [], []

print(f"\nStarting Adversarial Training (QSLP) for {num_epochs} epochs...")
print(f"Attacks: FGSM(ε={FGSM_EPS}) + PGD(ε={PGD_EPS}, α={PGD_ALPHA}, k={PGD_ITERS})")
print(f"Loss weights: clean={W_CLEAN} | QNI={W_QNI} | FGSM={W_FGSM} | PGD={W_PGD} | centroid={W_CENTROID}")
print("=" * 70)

for epoch in range(1, num_epochs + 1):

    # Recompute centroids every 5 epochs (Algorithm 1 lines 3-5)
    # Centroids drift as the model updates; refresh keeps QNI-CCP accurate
    if epoch % 5 == 0:
        print(f"  🔄 Recomputing centroids at epoch {epoch}...")
        centroids = compute_class_centroids(model, train_loader, device, num_classes)

    # One adversarial training epoch
    train_loss, train_acc = train_adversarial_epoch(
        model, train_loader, optimizer, centroids, device
    )

    # Standard clean validation
    val_loss, val_acc = evaluate(model, val_loader, device)

    # Scheduler step (reduce LR on plateau)
    scheduler.step(train_loss)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)

    print(f"Epoch [{epoch:02d}/{num_epochs}] | LR: {optimizer.param_groups[0]['lr']:.6f}")
    print(f"  Train Loss: {train_loss:.4f} | Train Acc (clean): {train_acc:.4f}")
    print(f"  Val   Loss: {val_loss:.4f}  | Val   Acc (clean): {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc               = val_acc
        epochs_without_improvement = 0
        torch.save({
            'epoch':                epoch,
            'model_state_dict':     model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc':              val_acc,
            'val_loss':             val_loss,
            'config': {
                'n_qubits':    n_qubits,
                'q_out_dim':   q_out_dim,
                'num_classes': num_classes,
                'adv_trained': True,
                'fgsm_eps':    FGSM_EPS,
                'pgd_eps':     PGD_EPS,
            }
        }, "adversarial_resnet_v3.pth")
        print(f"  💾 Best model saved (Val Acc: {best_val_acc:.4f})")
    else:
        epochs_without_improvement += 1
        print(f"  🕒 No improvement for {epochs_without_improvement} epoch(s).")

    if epochs_without_improvement >= early_stopping_patience:
        print(f"\n⏹️  Early stopping triggered at epoch {epoch}.")
        break

    print("-" * 70)

# ── STEP 5: Final adversarial robustness evaluation ──────────────────────────
print("\n" + "=" * 70)
print("FINAL ADVERSARIAL ROBUSTNESS EVALUATION")
print("=" * 70)

# Load the best checkpoint for evaluation
checkpoint = torch.load("adversarial_resnet_v3.pth", map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])

_, clean_acc      = evaluate(model, test_loader, device)
fgsm_acc, pgd_acc = evaluate_adversarial(model, test_loader, device)

print(f"\n  Clean Test Accuracy : {clean_acc:.4f}  ({clean_acc*100:.1f}%)")
print(f"  FGSM  Test Accuracy : {fgsm_acc:.4f}  ({fgsm_acc*100:.1f}%)  [ε={FGSM_EPS}]")
print(f"  PGD   Test Accuracy : {pgd_acc:.4f}  ({pgd_acc*100:.1f}%)  [ε={PGD_EPS}, k={PGD_ITERS}]")
print(f"\n  FGSM robustness drop : {(clean_acc - fgsm_acc)*100:.1f}%")
print(f"  PGD  robustness drop : {(clean_acc - pgd_acc)*100:.1f}%")
print("\n✅ Adversarial training complete.")

# ══════════════════════════════════════════════════════════════════════════════
# HOW TO INTERPRET RESULTS
# ══════════════════════════════════════════════════════════════════════════════
#
# COMPARE THREE CHECKPOINTS:
#   hybrid_resnet_v3.pth          → clean baseline
#   classical_resnet_v3.pth       → clean classical baseline
#   adversarial_resnet_v3.pth     → after adversarial training
#
# EXPECTED OUTCOME (from paper Table 5 / Figure 6):
#   • Clean acc may drop slightly (1-3%): model becomes more conservative
#   • FGSM acc should improve dramatically (e.g., 20% → 80%+)
#   • PGD  acc should improve most (paper: <10% → ~80% after QSLP)
#
# IF CLEAN ACC DROPS TOO MUCH (>5%):
#   • Increase W_CLEAN (e.g., 0.60) and reduce W_PGD (e.g., 0.10)
#   • Lower FGSM_EPS or PGD_EPS
#   • Reduce EPSILON_Q for gentler QNI-CCP perturbations
#
# IF ADVERSARIAL ACC IS STILL LOW:
#   • Increase PGD_ITERS (e.g., 10-20) for stronger training signal
#   • Increase W_PGD (e.g., 0.20) and reduce W_CLEAN (e.g., 0.40)
#   • Increase EPSILON_Q for stronger QNI-CCP perturbations
# ══════════════════════════════════════════════════════════════════════════════